In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time

import pandas as pd
import requests

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://api.open.fec.gov/v1/candidates/totals/"
STATES = ["MI", "TX", "OH", "IA"]
HEADERS = {
    "User-Agent": "WebDataScience/1.0 (fiona.kim@colorado.edu)"
}

In [ ]:
all_records = []
record_counts = {}

for state in STATES:
    params = {
        "api_key": "DEMO_KEY",
        "cycle": 2026,
        "office": "S",
        "state": state,
        "per_page": 100,
        "page": 1,
    }

    state_records = []

    while True:
        response = requests.get(
            API_URL,
            params=params,
            headers=HEADERS,
            timeout=30
        )
        response.raise_for_status()

        page_data = response.json()
        state_records.extend(page_data.get("results", []))

        total_pages = page_data.get("pagination", {}).get("pages", 1)
        if params["page"] >= total_pages:
            break

        params["page"] += 1
        time.sleep(1)

    all_records.extend(state_records)
    record_counts[state] = len(state_records)
    time.sleep(1)

collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_file = RAW_DIR / f"fec_2026_senate_{stamp}.json"

with raw_file.open("w") as file:
    json.dump(
        {
            "source": API_URL,
            "collected_at_utc": collected_at,
            "filters": {
                "cycle": 2026,
                "office": "S",
                "states": STATES
            },
            "record_counts_by_state": record_counts,
            "results": all_records
        },
        file,
        indent=2
    )

print("Records by state:", record_counts)
print("Total records:", len(all_records))
print("Saved raw data to:", raw_file)

In [ ]:
raw_df = pd.json_normalize(all_records)

print("Raw rows and columns:", raw_df.shape)
print(raw_df.columns.tolist())

display(raw_df.head())

In [ ]:
# First, define which columns to keep
columns_to_keep = [
    "candidate_id",
    "name",
    "party",
    "party_full",
    "state",
    "office",
    "receipts",
    "disbursements",
    "cash_on_hand_end_period",
    "debts_owed_by_committee",
    "coverage_start_date",
    "coverage_end_date",
    "last_file_date",
]

# Add missing columns to raw_df if they don't exist
for column in columns_to_keep:
    if column not in raw_df.columns:
        raw_df[column] = pd.NA

# Create the clean dataframe by selecting only the columns we want
clean = raw_df[columns_to_keep].copy()

# Rename columns for clarity
clean = clean.rename(columns={
    "name": "candidate_name",
    "party": "party_code",
    "party_full": "party_name",
    "cash_on_hand_end_period": "cash_on_hand",
    "debts_owed_by_committee": "debts_owed",
    "last_file_date": "last_filing_date",
})

# Convert money columns to numeric
money_columns = [
    "receipts",
    "disbursements",
    "cash_on_hand",
    "debts_owed",
]

for column in money_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

# Convert date columns to datetime
date_columns = [
    "coverage_start_date",
    "coverage_end_date",
    "last_filing_date",
]

for column in date_columns:
    clean[column] = pd.to_datetime(clean[column], errors="coerce")

# Remove rows with missing critical data and duplicates
clean = clean.dropna(subset=["candidate_id", "candidate_name"])
clean = clean.drop_duplicates(subset="candidate_id")

# Sort the data
clean = clean.sort_values(
    ["state", "party_name", "candidate_name"],
    na_position="last"
).reset_index(drop=True)

# Display summary statistics
print("Cleaned rows and columns:", clean.shape)
print("Rows by state:")
display(clean["state"].value_counts())
print("Missing values:")
display(clean.isna().sum().to_frame("missing"))

display(clean.head())

# NOW save the cleaned data (after 'clean' has been created and processed)
clean.to_csv(
    PROCESSED_DIR / "fec_2026_senate_clean.csv",
    index=False
)

# Create and save a sample
sample = (
    clean.groupby("state", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

sample.to_csv("data/sample_clean.csv", index=False)

print("Saved full cleaned data locally.")
print(f"Saved a {len(sample)}-row sample to data/sample_clean.csv")

In [ ]:
clean.to_csv(
    PROCESSED_DIR / "fec_2026_senate_clean.csv",
    index=False
)


sample = (
    clean.groupby("state", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

sample.to_csv("data/sample_clean.csv", index=False)

print("Saved full cleaned data locally.")
print(f"Saved a {len(sample)}-row sample to data/sample_clean.csv")

In [ ]:
print("Cleaned records by state:")
display(clean["state"].value_counts().sort_index())

print("Sample rows by state:")
display(sample["state"].value_counts().sort_index())

print("Duplicate candidate IDs:", clean["candidate_id"].duplicated().sum())
print("Missing receipts:", clean["receipts"].isna().sum())
print("Missing disbursements:", clean["disbursements"].isna().sum())